# NN1 — Feedforward Net, Rolling-Window Train/Valid/Test

Replicates the **NN1** architecture from Gu, Kelly & Xiu (2020), *Empirical Asset Pricing via Machine Learning*: one hidden layer (32 units), ReLU, batch-norm, He init, Adam, L1 weight penalty, early stopping on a validation block, and a 10-model ensemble averaged over random seeds.

**Rolling scheme:** 5 years train / 5 years validate / 1 year test, stepped forward one year at a time, from the first feasible test year (1994) through 2025 (partial, to March).

**Memory:** the panel is one `.parquet` file, but each window is read with a **year filter and column projection** pushed into the parquet reader, so only ~11 years of the needed feature columns ever enter RAM — never the full 2.4M-row panel.

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# ---- configuration ---------------------------------------------------------
PARQUET_PATH = "US_GFD_FEATURES.parquet"
OUTPUT_PATH  = "NN1_predictions.parquet"
MODEL_DIR    = "NN1_models"  # per-seed weights saved here for later variable importance

FIRST_YEAR = 1984      # first year present in the data
LAST_YEAR  = 2025      # last test year (2025 is partial, through March)
N_TRAIN    = 5         # years in the training block
N_VALID    = 5         # years in the validation block
N_TEST     = 1         # years in the test block (step size is also 1 year)

N_ENSEMBLE = 10        # models averaged per window (GKX default)
HIDDEN     = 32        # NN1: single hidden layer of 32 units
L1_LAMBDA  = 1e-5      # L1 penalty on weights
LR         = 1e-3
MAX_EPOCHS = 100
PATIENCE   = 5         # early-stopping patience on validation loss
BATCH_SIZE = 10_000

# columns that are NOT features (identifiers + target/price bookkeeping)
ID_COLS = ["permno","eom","gvkey","iid","cusip","tic","tpci","exchg",
           "shrcd","exchcd","sic","naics","trade_eom","acc_eom",
           "acc_datadate","rdq","date_buy_t1","date_sell_t1","ret_1m",
           "target","prc","prc_buy_t1","prc_sell_t1","n_days",
           "acc_age_m","target_w"]
TARGET = "target_w"

In [ ]:
# ---- resolve the feature list from the parquet schema (no data read) -------
schema = pq.read_schema(PARQUET_PATH)
all_cols = schema.names
FEATURES = [c for c in all_cols if c not in ID_COLS]
print(f"{len(FEATURES)} feature columns")

# columns we actually pull from disk: features + target + the three id cols we keep
READ_COLS = ["permno", "gvkey", "eom", TARGET] + FEATURES

In [ ]:
# ---- read the projected columns ONCE, then slice by year in memory ---------
# The file has only 3 row groups, so per-window filtered reads can't skip much
# and re-reading every window is what stalls. Pull the feature columns a single
# time and slice each window out of RAM.
import time

_t = time.time()
FULL = pq.read_table(PARQUET_PATH, columns=READ_COLS).to_pandas()

# eom often comes back as object/string from the parquet writer -> force datetime
FULL["eom"] = pd.to_datetime(FULL["eom"])

# cast features + target together in one shot (avoids a fragmented frame)
FULL = FULL.astype({**{c: "float32" for c in FEATURES}, TARGET: "float32"})
FULL["_year"] = FULL["eom"].dt.year
FULL = FULL.copy()   # de-fragment after the wide cast

print(f"read {len(FULL):,} rows x {len(FULL.columns)} cols in "
      f"{time.time()-_t:.1f}s, "
      f"{FULL.memory_usage(deep=True).sum()/1e9:.2f} GB in memory")

# ---- data-health audit: catch the things that make validation loss NaN -----
_feat_na  = FULL[FEATURES].isna().to_numpy().sum()
_feat_inf = np.isinf(FULL[FEATURES].to_numpy()).sum()
_targ_na  = FULL[TARGET].isna().sum()
print(f"feature NaNs: {_feat_na:,} | feature Infs: {_feat_inf:,} | "
      f"target NaNs: {_targ_na:,}")
if _feat_na or _feat_inf:
    print("  -> imputing residual feature NaN/Inf to 0.5 (median rank)")
    FULL[FEATURES] = FULL[FEATURES].replace([np.inf, -np.inf], np.nan)
    FULL[FEATURES] = FULL[FEATURES].fillna(0.5).astype("float32")

def slice_years(y0, y1):
    """Rows whose eom-year is in [y0, y1], sliced from the in-memory frame.
    Rows with a missing target are dropped -- they can't train or score."""
    sub = FULL[FULL["_year"].between(y0, y1)]
    return sub[sub[TARGET].notna()]

In [ ]:
# ---- rolling window schedule ----------------------------------------------
def make_windows(first_year, last_year, n_tr, n_va, n_te):
    """Yield (train_years, valid_years, test_years) tuples, stepping 1 year.
    Blocks are strictly time-ordered: train < valid < test, no overlap."""
    wins = []
    test_start = first_year + n_tr + n_va
    while test_start <= last_year:
        tr = (test_start - n_tr - n_va, test_start - n_va - 1)
        va = (test_start - n_va,        test_start - 1)
        te = (test_start,               min(test_start + n_te - 1, last_year))
        wins.append((tr, va, te))
        test_start += n_te
    return wins

WINDOWS = make_windows(FIRST_YEAR, LAST_YEAR, N_TRAIN, N_VALID, N_TEST)
print(f"{len(WINDOWS)} windows")
print("first:", WINDOWS[0])
print("last :", WINDOWS[-1])

In [ ]:
# ---- NN1 architecture (GKX) -----------------------------------------------
class NN1(nn.Module):
    """One hidden layer, 32 units, ReLU, batch-norm, single linear output."""
    def __init__(self, n_features, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )
        # He (Kaiming) initialization on the linear layers
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [ ]:
# ---- train one model with early stopping ----------------------------------
def train_one(Xtr, ytr, Xva, yva, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = NN1(Xtr.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    # force float32 tensors: if features are stored float16, half-precision
    # through BatchNorm/ReLU (esp. on CPU) overflows to NaN and training stalls
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device)
    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=device)
    yva_t = torch.tensor(yva, dtype=torch.float32, device=device)

    n = Xtr_t.shape[0]
    best_val, best_state, wait = float("inf"), None, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=device)   # shuffle WITHIN train block only
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i+BATCH_SIZE]
            opt.zero_grad()
            pred = model(Xtr_t[idx])
            loss = loss_fn(pred, ytr_t[idx])
            l1 = sum(p.abs().sum() for p in model.parameters())
            (loss + L1_LAMBDA * l1).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val = loss_fn(model(Xva_t), yva_t).item()

        # a NaN/inf val loss never beats best_val -> best_state stays None and
        # load_state_dict(None) crashes. Surface the real cause instead.
        if not np.isfinite(val):
            raise FloatingPointError(
                f"validation loss is {val} at epoch {epoch} (seed {seed}). "
                "Check features for NaN/Inf or lower LR.")

        if val < best_val:
            best_val = val
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state if best_state is not None else model.state_dict())
    return model

In [ ]:
# ---- predict with an ensemble of seeded models ----------------------------
def predict_ensemble(models, X):
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    preds = np.zeros((len(models), X.shape[0]), dtype=np.float32)
    for j, m in enumerate(models):
        m.eval()
        with torch.no_grad():
            preds[j] = m(Xt).cpu().numpy()
    return preds.mean(axis=0)

In [ ]:
# ---- roll through windows, write predictions incrementally ----------------
import pyarrow as pa
import json
from pathlib import Path

# folder for saved weights + the feature order (needed to map weights/importance
# back to factor names later -- a state_dict only stores numbered tensors).
Path(MODEL_DIR).mkdir(exist_ok=True)
json.dump(FEATURES, open(f"{MODEL_DIR}/feature_names.json", "w"))

writer = None
for (tr, va, te) in WINDOWS:
    # read only the years this window needs (train+valid together, test separately)
    train_df = slice_years(tr[0], va[1])          # 10 yrs: train+valid span
    test_df  = slice_years(te[0], te[1])          # 1 test yr

    tr_mask = train_df["eom"].dt.year.between(*tr)
    va_mask = train_df["eom"].dt.year.between(*va)

    Xtr = train_df.loc[tr_mask, FEATURES].to_numpy(np.float32)
    ytr = train_df.loc[tr_mask, TARGET].to_numpy(np.float32)
    Xva = train_df.loc[va_mask, FEATURES].to_numpy(np.float32)
    yva = train_df.loc[va_mask, TARGET].to_numpy(np.float32)
    Xte = test_df[FEATURES].to_numpy(np.float32)

    # train the ensemble on this window, saving each seed's weights as we go
    models = []
    for s in range(N_ENSEMBLE):
        m = train_one(Xtr, ytr, Xva, yva, seed=s)
        torch.save(m.state_dict(), f"{MODEL_DIR}/nn1_test{te[0]}_seed{s}.pt")
        models.append(m)
    yhat = predict_ensemble(models, Xte)

    out = test_df[["permno", "gvkey", "eom", TARGET]].copy()
    out["prediction"] = yhat

    # stream this test year's predictions to disk; don't accumulate in RAM
    tbl = pa.Table.from_pandas(out, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, tbl.schema)
    writer.write_table(tbl)

    print(f"test {te[0]}: train {tr}, valid {va} | "
          f"n_train={len(Xtr):,} n_test={len(Xte):,} | "
          f"pred mean={yhat.mean():.5f}")

    # free the window before the next iteration
    del Xtr, ytr, Xva, yva, Xte, models
    if device.type == "cuda":
        torch.cuda.empty_cache()

if writer is not None:
    writer.close()
print("done ->", OUTPUT_PATH)

In [ ]:
# ---- inspect the saved predictions ----------------------------------------
preds = pd.read_parquet(OUTPUT_PATH)
print(preds.shape)
print(preds.columns.tolist())
preds.head()

In [ ]:
# ---- inspect saved model weights -------------------------------------------
import json
FEATURES = json.load(open("NN1_models/feature_names.json"))
model = NN1(len(FEATURES)).to(device)
model.load_state_dict(torch.load("NN1_models/nn1_test2010_seed3.pt"))
model.eval()

In [ ]:
# ---- permutation variable importance ---------------------------------------
import json, numpy as np, pandas as pd, torch
from pathlib import Path

def permutation_importance(test_df, test_year, model_dir="NN1_models",
                           n_repeats=5, seed=0):
    """Ensemble permutation importance for one test year.

    Reloads all 10 seeds for that year, averages their predictions, then for
    each feature shuffles that column and measures how much MSE rises vs the
    real target. Bigger rise = the model leaned on that feature more.
    `test_df` must contain the FEATURES columns and 'target_w' for that year.
    """
    FEATURES = json.load(open(Path(model_dir) / "feature_names.json"))
    X = test_df[FEATURES].to_numpy(np.float32)
    y = test_df["target_w"].to_numpy(np.float32)
    yt = torch.tensor(y, device=device)

    # reload every seed for this test year
    models = []
    for p in sorted(Path(model_dir).glob(f"nn1_test{test_year}_seed*.pt")):
        m = NN1(len(FEATURES)).to(device)
        m.load_state_dict(torch.load(p)); m.eval()
        models.append(m)

    def ens_pred(arr):
        Xt = torch.tensor(arr, device=device)
        with torch.no_grad():
            return torch.stack([m(Xt) for m in models]).mean(0)

    base = torch.mean((ens_pred(X) - yt) ** 2).item()
    rng = np.random.default_rng(seed)
    rows = []
    for j, f in enumerate(FEATURES):
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[:, j] = Xp[rng.permutation(len(Xp)), j]   # shuffle feature j
            drops.append(torch.mean((ens_pred(Xp) - yt) ** 2).item() - base)
        rows.append({"feature": f, "importance": np.mean(drops), "std": np.std(drops)})
    return (pd.DataFrame(rows)
            .sort_values("importance", ascending=False)
            .reset_index(drop=True))

In [ ]:
test_df = slice_years(2010, 2010)
imp = permutation_importance(test_df, 2010)
print(imp.head(20))

In [ ]:
import matplotlib.pyplot as plt

TOP_N = 40   # show the top N; set to len(imp) for all features

d = imp.head(TOP_N).iloc[::-1]   # reverse: largest ends up at the top of barh

fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(d))))
ax.barh(d["feature"], d["importance"], xerr=d["std"],
        color="steelblue", edgecolor="black", linewidth=0.5,
        error_kw={"lw": 0.8})
ax.set_xlabel("permutation importance (rise in MSE when shuffled)")
ax.set_title(f"NN1 variable importance — top {TOP_N}")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()